# SNP embedding -> gene expression difference

This notebook prepares train/validation/test splits for the 101 human samples, builds reference artifacts from the 81 training samples, and trains a three-layer linear model. It is intentionally unexecuted.

Inputs:
- SNP embeddings: `/mnt/a100-nas-new/peixunban/tanxinjiang/13.SNPbag.pre_exp/model_training/embeddings_gaussian_sigma15.0`
- VCF directory: `/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/101vcf`
- Target genes: `/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/test.10gene.bed`
- Reference builders: `/mnt/rice/default/Workspace/xuxiaolong/human/average_bigwig.py` and `/mnt/rice/default/Workspace/xuxiaolong/human/make_major_allele_consensus.sh`


In [1]:
from __future__ import annotations

import bisect
import gc
import json
import math
import os
import re
import subprocess
import warnings
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import pysam
except ImportError as exc:
    raise ImportError('Install pysam before running this notebook: pip install pysam') from exc

try:
    import pyBigWig
except ImportError as exc:
    raise ImportError('Install pyBigWig before running this notebook: pip install pyBigWig') from exc

try:
    import swanlab
except ImportError:
    swanlab = None

In [ ]:
PROJECT_DIR = Path('/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding')
HUMAN_DIR = Path('/mnt/rice/default/Workspace/xuxiaolong/human')

EMBEDDING_DIR = Path('/mnt/a100-nas-new/peixunban/tanxinjiang/13.SNPbag.pre_exp/model_training/embeddings_gaussian_sigma15.0')
VCF_DIR = PROJECT_DIR / '101vcf'  # fallback: per-sample symlinked CIMA-Hxxx_CIMA-Hxxx.vcf.gz files
COMBINED_SNP_VCF = HUMAN_DIR / 'CIMA_consensus' / 'maf01con.only_snp.vcf.gz'  # preferred 413-sample SNP-only merged VCF
GENE_BED = PROJECT_DIR / 'test.10gene.bed'
GENCODE_GTF = HUMAN_DIR / 'gencode.v49.annotation.sorted.gtf'
SAMPLE_NAME_FILE = PROJECT_DIR / '101samples.name.txt'

REFERENCE_FASTA = HUMAN_DIR / 'hg38_forCIMA.fa'
BIGWIG_DIR = Path('/mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101')
BIGWIG_PATTERN = '*/re-normalized_Monocyte.total.bw'

REFERENCE_DIR = PROJECT_DIR / 'reference_from_train81'
REFERENCE_PREFIX = REFERENCE_DIR / 'train81.major'
REFERENCE_AVERAGE_BW = REFERENCE_DIR / 'train81.average.bw'
REFERENCE_MAJOR_TSV = REFERENCE_DIR / 'train81.major.major_allele.tsv'
REFERENCE_CONSENSUS_FA = REFERENCE_DIR / 'train81.major.major_consensus.fa'

MICROMAMBA_ENV = 'tools'
MICROMAMBA_CACHE_HOME = Path('/tmp/codex-mamba-cache')

def micromamba_cmd(cmd: list[str]) -> list[str]:
    # XDG_CACHE_HOME avoids lockfile errors when the home mamba cache is read-only.
    return ['env', f'XDG_CACHE_HOME={MICROMAMBA_CACHE_HOME}', 'micromamba', 'run', '-n', MICROMAMBA_ENV, *map(str, cmd)]

N_SNPS = 1024
EMBEDDING_DIM = 512  # Optional expected embedding width; keep None to infer from cached/loaded embeddings.
VARIANT_EMBEDDING_MODE = 'alt_minus_ref'  # For dict entries with emb_ref/emb_alt: use 'alt_minus_ref', 'alt', or 'ref'.
GENE_EMBEDDING_CACHE_DIR = PROJECT_DIR / 'gene_embedding_windows'
REBUILD_GENE_EMBEDDING_CACHE = False
TARGET_MODE = 'three_prime_exon'  # target is mean bigWig signal over the transcript's most 3-prime exon.
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_SWANLAB = True
SWANLAB_API_KEY = os.environ.get('SWANLAB_API_KEY')
SWANLAB_PROJECT = 'snp_embedding_expression_diff'
SWANLAB_EXPERIMENT_NAME = 'three_layer_regressor'
SWANLAB_MODE = 'online'  # Use 'online' after `swanlab login`; use 'disabled' to turn logging off.
SWANLAB_LOG_PER_GENE_VAL = True

VAL_INDIVIDUALS = ['H005', 'H010', 'H055', 'H102', 'H103', 'H137', 'H198', 'H202', 'H276', 'H319']
TEST_INDIVIDUALS = ['H030', 'H117', 'H118', 'H129', 'H195', 'H197', 'H215', 'H225', 'H261', 'H309']

In [3]:
def read_sample_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t', header=None, names=['accession', 'sample'])
    df['accession'] = df['accession'].astype(str)
    df['sample'] = df['sample'].astype(str)
    return df

sample_table = read_sample_table(SAMPLE_NAME_FILE)
sample_to_accession = dict(zip(sample_table['sample'], sample_table['accession']))
all_individuals = sample_table['sample'].tolist()

def bigwig_path_for_sample(sample: str) -> Path:
    accession = sample_to_accession[sample]
    return BIGWIG_DIR / accession / 're-normalized_Monocyte.total.bw'

def find_bigwig_path(sample: str) -> Path | None:
    path = bigwig_path_for_sample(sample)
    return path if path.exists() else None

val_set = set(VAL_INDIVIDUALS)
test_set = set(TEST_INDIVIDUALS)
train_individuals = [x for x in all_individuals if x not in val_set and x not in test_set]

assert not (val_set & test_set), 'Validation and test sets overlap.'
assert set(VAL_INDIVIDUALS).issubset(all_individuals), 'Some validation individuals are missing from sample table.'
assert set(TEST_INDIVIDUALS).issubset(all_individuals), 'Some test individuals are missing from sample table.'
assert len(train_individuals) == 81, f'Expected 81 training samples, got {len(train_individuals)}.'

missing_bigwig = [sample for sample in all_individuals if find_bigwig_path(sample) is None]
assert not missing_bigwig, f'Samples missing normalized bigWig: {missing_bigwig}'
expression_train_individuals = train_individuals

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
(REFERENCE_DIR / 'train_individuals.txt').write_text('\n'.join(train_individuals) + '\n')
(REFERENCE_DIR / 'train_expression_individuals.txt').write_text('\n'.join(str(bigwig_path_for_sample(s)) for s in expression_train_individuals) + '\n')
(REFERENCE_DIR / 'val_individuals.txt').write_text('\n'.join(VAL_INDIVIDUALS) + '\n')
(REFERENCE_DIR / 'test_individuals.txt').write_text('\n'.join(TEST_INDIVIDUALS) + '\n')

print(f'genotype_train={len(train_individuals)} expression_train={len(expression_train_individuals)} val={len(VAL_INDIVIDUALS)} test={len(TEST_INDIVIDUALS)}')
print(f'Using normalized bigWigs under: {BIGWIG_DIR}')

genotype_train=81 expression_train=81 val=10 test=10
Using normalized bigWigs under: /mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101


## Build reference from the 81 training samples

The preferred VCF is `/mnt/rice/default/Workspace/xuxiaolong/human/CIMA_consensus/maf01con.only_snp.vcf.gz`: a 413-sample merged VCF that has already been filtered with `bcftools view -v snps`. It contains all 101 samples used here, so the notebook can pass it directly to `make_major_allele_consensus.sh -s train_individuals.txt` and avoid merging the 81 per-sample VCFs. The original per-sample VCFs remain a fallback only.

Expression bigWigs are read from `/mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101/<accession>/re-normalized_Monocyte.total.bw`, using the accession-to-sample mapping in `101samples.name.txt`.


In [4]:
def find_sample_vcf(sample: str, vcf_dir: Path = VCF_DIR) -> Path:
    candidates = [
        vcf_dir / f'CIMA-{sample}_CIMA-{sample}.vcf.gz',
        vcf_dir / f'{sample}.vcf.gz',
    ]
    candidates.extend(sorted(vcf_dir.glob(f'*{sample}*.vcf.gz')))
    existing = []
    seen = set()
    for path in candidates:
        # Keep valid symlinks and symlink paths themselves; targets may live on another mount.
        if (path.exists() or path.is_symlink()) and path not in seen:
            existing.append(path)
            seen.add(path)
    if len(existing) != 1:
        raise FileNotFoundError(f'Expected one VCF for {sample}, found {len(existing)}: {existing[:5]}')
    return existing[0]

def assert_samples_in_combined_vcf(vcf_path: Path, samples: list[str]) -> None:
    vcf = pysam.VariantFile(str(vcf_path))
    try:
        available = set(vcf.header.samples)
    finally:
        vcf.close()
    missing = [sample for sample in samples if not ({sample, f'CIMA-{sample}', f'CIMA-{sample}_CIMA-{sample}'} & available)]
    if missing:
        raise ValueError(f'{len(missing)} samples are missing from {vcf_path}: {missing[:20]}')

assert COMBINED_SNP_VCF.exists(), COMBINED_SNP_VCF
assert (Path(str(COMBINED_SNP_VCF) + '.tbi').exists() or Path(str(COMBINED_SNP_VCF) + '.csi').exists()), 'Combined SNP VCF needs a tabix/csi index.'
assert_samples_in_combined_vcf(COMBINED_SNP_VCF, all_individuals)
print(f'Using SNP-only merged VCF: {COMBINED_SNP_VCF}')

Using SNP-only merged VCF: /mnt/rice/default/Workspace/xuxiaolong/human/CIMA_consensus/maf01con.only_snp.vcf.gz


In [5]:
major_cmd = micromamba_cmd([
    'bash', str(HUMAN_DIR / 'make_major_allele_consensus.sh'),
    '-r', str(REFERENCE_FASTA),
    '-v', str(COMBINED_SNP_VCF),
    '-o', str(REFERENCE_PREFIX),
    '-s', str(REFERENCE_DIR / 'train_individuals.txt'),  # genotype reference uses all 81 training samples
    '--biallelic-only',
])

average_bw_cmd = micromamba_cmd([
    'python3', str(HUMAN_DIR / 'average_bigwig.py'),
    '-i', str(BIGWIG_DIR),
    '-o', str(REFERENCE_AVERAGE_BW),
    '--pattern', BIGWIG_PATTERN,
    '--sample-list', str(REFERENCE_DIR / 'train_expression_individuals.txt'),  # expression reference uses samples with bigWig
    '--force',
])

print('Major allele consensus command, using existing SNP-only merged VCF:')
print(' '.join(map(str, major_cmd)))
print('\nAverage bigWig command:')
print(' '.join(map(str, average_bw_cmd)))

# Fixed to micromamba env `tools`; uncomment to run after paths are checked:
# subprocess.run(major_cmd, check=True)
# subprocess.run(average_bw_cmd, check=True)

Major allele consensus command, using existing SNP-only merged VCF:
env XDG_CACHE_HOME=/tmp/codex-mamba-cache micromamba run -n tools bash /mnt/rice/default/Workspace/xuxiaolong/human/make_major_allele_consensus.sh -r /mnt/rice/default/Workspace/xuxiaolong/human/hg38_forCIMA.fa -v /mnt/rice/default/Workspace/xuxiaolong/human/CIMA_consensus/maf01con.only_snp.vcf.gz -o /mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/reference_from_train81/train81.major -s /mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/reference_from_train81/train_individuals.txt --biallelic-only

Average bigWig command:
env XDG_CACHE_HOME=/tmp/codex-mamba-cache micromamba run -n tools python3 /mnt/rice/default/Workspace/xuxiaolong/human/average_bigwig.py -i /mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101 -o /mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/reference_from_train81/train81.average.bw --pattern */re-normalized_Monocyte.total.bw --sample-list /mnt/rice/default/Workspace/xuxiaolong/

In [6]:
@dataclass(frozen=True)
class GeneRecord:
    chrom: str
    start: int
    end: int
    name: str
    strand: str
    tss: int
    transcript_id: str
    expression_regions: tuple[tuple[str, int, int], ...]


def normalize_chrom(chrom: str) -> str:
    chrom = str(chrom)
    return chrom if chrom.startswith('chr') else f'chr{chrom}'


def chrom_number(chrom: str) -> str:
    return normalize_chrom(chrom).replace('chr', '')


def strip_ensembl_version(value: str) -> str:
    return str(value).split('.', 1)[0]


def transcript_id_from_gene_name(name: str) -> str:
    parts = str(name).split('|')
    for part in parts:
        if part.startswith('ENST'):
            return part
    raise ValueError(f'Could not extract ENST transcript id from gene name: {name}')


def parse_gtf_attributes(attr_text: str) -> dict[str, str]:
    attrs = {}
    for key, value in re.findall(r'(\S+) "([^"]*)"', attr_text):
        attrs[key] = value
    return attrs


def load_transcript_exons(gtf_path: Path, transcript_ids: Iterable[str]) -> dict[str, list[tuple[str, int, int, str]]]:
    requested = {str(tid) for tid in transcript_ids}
    requested_base = {strip_ensembl_version(tid) for tid in requested}
    exons: dict[str, list[tuple[str, int, int, str]]] = {tid: [] for tid in requested}
    resolved: dict[str, str] = {tid: tid for tid in requested}

    with Path(gtf_path).open() as handle:
        for line in handle:
            if not line or line.startswith('#'):
                continue
            fields = line.rstrip('\n').split('\t')
            if len(fields) != 9 or fields[2] != 'exon':
                continue
            attrs = parse_gtf_attributes(fields[8])
            transcript_id = attrs.get('transcript_id')
            if not transcript_id:
                continue
            transcript_base = strip_ensembl_version(transcript_id)
            if transcript_id in requested:
                key = transcript_id
            elif transcript_base in requested_base:
                matches = [tid for tid in requested if strip_ensembl_version(tid) == transcript_base]
                if not matches:
                    continue
                key = matches[0]
                resolved[key] = transcript_id
            else:
                continue

            chrom = normalize_chrom(fields[0])
            start0 = int(fields[3]) - 1  # GTF is 1-based closed; pyBigWig uses 0-based half-open.
            end0 = int(fields[4])
            strand = fields[6]
            exons.setdefault(key, []).append((chrom, start0, end0, strand))

    missing = [tid for tid in requested if not exons.get(tid)]
    if missing:
        raise KeyError(f'No exon records found in {gtf_path} for transcript(s): {missing[:10]}')
    return exons


def select_three_prime_exon(exons: list[tuple[str, int, int, str]]) -> tuple[tuple[str, int, int], ...]:
    if not exons:
        raise ValueError('Cannot select 3-prime exon from an empty exon list.')
    strands = {exon[3] for exon in exons}
    if len(strands) != 1:
        raise ValueError(f'Expected one strand per transcript, got {sorted(strands)}')
    strand = next(iter(strands))
    if strand == '+':
        exon = max(exons, key=lambda item: (item[2], item[1]))
    elif strand == '-':
        exon = min(exons, key=lambda item: (item[1], item[2]))
    else:
        raise ValueError(f'Unsupported transcript strand: {strand!r}')
    chrom, start, end, _ = exon
    if end <= start:
        raise ValueError(f'Invalid exon interval for 3-prime exon: {exon}')
    return ((chrom, start, end),)


def read_gene_bed(path: Path, gtf_path: Path) -> list[GeneRecord]:
    raw_records = []
    with path.open() as handle:
        for line in handle:
            if not line.strip() or line.startswith('#'):
                continue
            fields = line.rstrip('\n').split('\t')
            chrom, start, end, name = fields[:4]
            strand = fields[5] if len(fields) > 5 else '+'
            start_i, end_i = int(start), int(end)
            tss = (start_i + end_i) // 2
            transcript_id = transcript_id_from_gene_name(name)
            raw_records.append((normalize_chrom(chrom), start_i, end_i, name, strand, tss, transcript_id))

    transcript_exons = load_transcript_exons(gtf_path, [record[6] for record in raw_records])
    genes: list[GeneRecord] = []
    for chrom, start_i, end_i, name, strand, tss, transcript_id in raw_records:
        exons = transcript_exons[transcript_id]
        expression_regions = select_three_prime_exon(exons)
        exon_strand = exons[0][3]
        genes.append(GeneRecord(chrom, start_i, end_i, name, exon_strand, tss, transcript_id, expression_regions))
    return genes


genes = read_gene_bed(GENE_BED, GENCODE_GTF)
gene_region_summary = pd.DataFrame([
    {
        'gene': g.name,
        'transcript_id': g.transcript_id,
        'strand': g.strand,
        'expression_regions': ';'.join(f'{chrom}:{start}-{end}' for chrom, start, end in g.expression_regions),
        'selected_bp': sum(end - start for _, start, end in g.expression_regions),
    }
    for g in genes
])
gene_region_summary


,chrom,start,end,name,strand,tss
0,chr4,150599627,151599627,RPS3A|ENST00000274065.9|win_21,+,151099627
1,chr5,147378710,148378710,SCGB3A2|ENST00000296694.5|win_24,+,147878710
2,chr8,143078338,144078338,NAPRT|ENST00000340490.7|win_45,-,143578338
3,chr9,136688675,137688675,SSNA1|ENST00000322310.10|win_52,+,137188675
4,chr12,7623616,8623616,CLEC4A|ENST00000229332.12|win_63,+,8123616
5,chr15,63651714,64651714,SNX22|ENST00000557789.5|win_76,+,64151714
6,chr17,7431998,8431998,TRAPPC1|ENST00000540486.5|win_84,-,7931998
7,chr17,38700506,39700506,RPL19|ENST00000678573.1|win_87,+,39200506
8,chr19,3072943,4072943,HMG20B|ENST00000333651.11|win_91,+,3572943
9,chr20,37027731,38027731,BLCAP|ENST00000397137.5|win_98,-,37527731


In [7]:
def variant_embedding_from_entry(entry: dict) -> torch.Tensor:
    emb_ref = torch.as_tensor(entry['emb_ref'], dtype=torch.float32).cpu()
    emb_alt = torch.as_tensor(entry['emb_alt'], dtype=torch.float32).cpu()
    if VARIANT_EMBEDDING_MODE == 'alt_minus_ref':
        return emb_alt - emb_ref
    if VARIANT_EMBEDDING_MODE == 'alt':
        return emb_alt
    if VARIANT_EMBEDDING_MODE == 'ref':
        return emb_ref
    raise ValueError(f'Unknown VARIANT_EMBEDDING_MODE={VARIANT_EMBEDDING_MODE!r}')

def variant_key(ref: str, alt: str) -> str:
    return f'{str(ref).upper()}>{str(alt).upper()}'

def unpack_variant_embedding_dict(payload: dict) -> tuple[np.ndarray, torch.Tensor, dict[int, dict[str, torch.Tensor]]]:
    variants_by_pos: dict[int, dict[str, torch.Tensor]] = {}
    skipped = 0
    for key, entry in payload.items():
        if not isinstance(entry, dict) or 'pos' not in entry or 'ref' not in entry or 'alt' not in entry or 'emb_ref' not in entry or 'emb_alt' not in entry:
            skipped += 1
            continue
        pos = int(entry['pos'])
        ref = str(entry['ref']).upper()
        alt = str(entry['alt']).upper()
        variants_by_pos.setdefault(pos, {})[variant_key(ref, alt)] = variant_embedding_from_entry(entry)

    if not variants_by_pos:
        raise KeyError('No variant entries with pos/emb_ref/emb_alt were found in embedding payload.')
    positions = np.asarray(sorted(variants_by_pos), dtype=np.int64)
    position_embeddings = torch.stack([next(iter(variants_by_pos[int(pos)].values())) for pos in positions]).contiguous()
    duplicate_variants = sum(max(0, len(v) - 1) for v in variants_by_pos.values())
    if skipped or duplicate_variants:
        warnings.warn(f'Embedding payload skipped {skipped} malformed entries and kept {duplicate_variants} additional REF/ALT entries at duplicate positions.')
    return positions, position_embeddings, variants_by_pos

def unpack_embedding_payload(payload):
    """Return (positions, embeddings) from common .pt layouts."""
    if isinstance(payload, dict):
        if payload:
            first_entry = next(iter(payload.values()))
            if isinstance(first_entry, dict) and {'pos', 'emb_ref', 'emb_alt'}.issubset(first_entry):
                positions, embeddings, _ = unpack_variant_embedding_dict(payload)
                return positions, embeddings
        emb = None
        pos = None
        for key in ('embeddings', 'embedding', 'x', 'features', 'snp_embeddings'):
            if key in payload:
                emb = payload[key]
                break
        for key in ('positions', 'pos', 'snp_positions', 'bp', 'base_positions'):
            if key in payload:
                pos = payload[key]
                break
        if emb is None or pos is None:
            raise KeyError(f'Could not find embeddings/positions keys. Available keys: {list(payload.keys())}')
    elif isinstance(payload, (tuple, list)) and len(payload) >= 2:
        first, second = payload[0], payload[1]
        if torch.as_tensor(first).ndim == 1:
            pos, emb = first, second
        else:
            emb, pos = first, second
    else:
        raise TypeError('Embedding .pt must be a dict or a tuple/list containing positions and embeddings.')

    pos = torch.as_tensor(pos, dtype=torch.long).cpu().numpy()
    emb = torch.as_tensor(emb, dtype=torch.float32).cpu()
    order = np.argsort(pos)
    return pos[order], emb[torch.as_tensor(order, dtype=torch.long)]

@lru_cache(maxsize=1)
def load_chrom_embeddings(chrom: str) -> tuple[np.ndarray, torch.Tensor]:
    chrom_id = chrom_number(chrom)
    path = EMBEDDING_DIR / f'1KGP.chr{chrom_id}.snps.embeddings.pt'
    if not path.exists():
        raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu')
    return unpack_embedding_payload(payload)

@lru_cache(maxsize=1)
def load_chrom_variant_embeddings(chrom: str) -> tuple[np.ndarray, torch.Tensor, dict[int, dict[str, torch.Tensor]] | None]:
    chrom_id = chrom_number(chrom)
    path = EMBEDDING_DIR / f'1KGP.chr{chrom_id}.snps.embeddings.pt'
    if not path.exists():
        raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu')
    if isinstance(payload, dict) and payload:
        first_entry = next(iter(payload.values()))
        if isinstance(first_entry, dict) and {'pos', 'ref', 'alt', 'emb_ref', 'emb_alt'}.issubset(first_entry):
            return unpack_variant_embedding_dict(payload)
    positions, embeddings = unpack_embedding_payload(payload)
    return positions, embeddings, None

def centered_snp_indices(positions: np.ndarray, tss: int, n_snps: int = N_SNPS) -> np.ndarray:
    center = bisect.bisect_left(positions, tss)
    left = center - n_snps // 2
    right = left + n_snps
    if left < 0:
        left = 0
        right = n_snps
    if right > len(positions):
        right = len(positions)
        left = max(0, right - n_snps)
    idx = np.arange(left, right)
    if len(idx) != n_snps:
        raise ValueError(f'Only found {len(idx)} SNPs around TSS={tss}; need {n_snps}.')
    return idx

def safe_filename(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def gene_embedding_cache_path(gene: GeneRecord, n_snps: int = N_SNPS) -> Path:
    gene_token = safe_filename(gene.name)[:120]
    filename = f'{gene.chrom}_{gene.tss}_{n_snps}_{gene_token}.pt'
    return GENE_EMBEDDING_CACHE_DIR / filename

def valid_gene_embedding_cache(cache_path: Path, n_snps: int = N_SNPS) -> bool:
    if not cache_path.exists():
        return False
    try:
        payload = torch.load(cache_path, map_location='cpu')
        positions = torch.as_tensor(payload['positions'])
        embeddings = torch.as_tensor(payload['embeddings'])
        variants_by_pos = payload.get('variants_by_pos')
        return positions.ndim == 1 and embeddings.ndim == 2 and len(positions) == n_snps and embeddings.shape[0] == n_snps and isinstance(variants_by_pos, dict)
    except Exception:
        return False

@lru_cache(maxsize=None)
def load_gene_embedding_window(gene: GeneRecord, n_snps: int = N_SNPS) -> tuple[np.ndarray, torch.Tensor, dict[int, dict[str, torch.Tensor]]]:
    cache_path = gene_embedding_cache_path(gene, n_snps)
    if valid_gene_embedding_cache(cache_path, n_snps) and not REBUILD_GENE_EMBEDDING_CACHE:
        payload = torch.load(cache_path, map_location='cpu')
        positions = torch.as_tensor(payload['positions'], dtype=torch.long).cpu().numpy()
        embeddings = torch.as_tensor(payload['embeddings'], dtype=torch.float32).cpu()
        variants_by_pos = payload.get('variants_by_pos') or {}
        if len(positions) != n_snps:
            raise ValueError(f'Cached window {cache_path} has {len(positions)} SNPs; expected {n_snps}.')
        return positions, embeddings, variants_by_pos

    positions, embeddings, variants_by_pos = load_chrom_variant_embeddings(gene.chrom)
    idx = centered_snp_indices(positions, gene.tss, n_snps)
    selected_positions = np.asarray(positions[idx], dtype=np.int64)
    selected_embeddings = embeddings[idx].clone().contiguous().cpu()
    selected_variants_by_pos = {}
    if variants_by_pos is not None:
        selected_variants_by_pos = {int(pos): variants_by_pos.get(int(pos), {}) for pos in selected_positions}
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'chrom': gene.chrom,
        'gene': gene.name,
        'tss': gene.tss,
        'n_snps': n_snps,
        'positions': torch.as_tensor(selected_positions, dtype=torch.long),
        'embeddings': selected_embeddings,
        'variants_by_pos': selected_variants_by_pos,
    }, cache_path)
    return selected_positions, selected_embeddings, selected_variants_by_pos

def prepare_gene_embedding_cache(genes: list[GeneRecord], n_snps: int = N_SNPS):
    GENE_EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    genes_by_chrom: dict[str, list[GeneRecord]] = {}
    for gene in genes:
        genes_by_chrom.setdefault(gene.chrom, []).append(gene)

    total = len(genes)
    done = 0
    for chrom, chrom_genes in sorted(genes_by_chrom.items()):
        missing_genes = [
            gene for gene in chrom_genes
            if REBUILD_GENE_EMBEDDING_CACHE or not valid_gene_embedding_cache(gene_embedding_cache_path(gene, n_snps), n_snps)
        ]
        if not missing_genes:
            for gene in chrom_genes:
                done += 1
                cache_path = gene_embedding_cache_path(gene, n_snps)
                print(f'[{done}/{total}] cached: {gene.name} -> {cache_path}', flush=True)
            continue

        print(f'Loading full embedding for {chrom} to extract {len(missing_genes)} gene window(s)...', flush=True)
        positions, embeddings, variants_by_pos = load_chrom_variant_embeddings(chrom)
        try:
            for gene in chrom_genes:
                done += 1
                cache_path = gene_embedding_cache_path(gene, n_snps)
                if valid_gene_embedding_cache(cache_path, n_snps) and not REBUILD_GENE_EMBEDDING_CACHE:
                    print(f'[{done}/{total}] cached: {gene.name} -> {cache_path}', flush=True)
                    continue
                idx = centered_snp_indices(positions, gene.tss, n_snps)
                selected_positions = np.asarray(positions[idx], dtype=np.int64)
                selected_embeddings = embeddings[idx].clone().contiguous().cpu()
                selected_variants_by_pos = {}
                if variants_by_pos is not None:
                    selected_variants_by_pos = {int(pos): variants_by_pos.get(int(pos), {}) for pos in selected_positions}
                tmp_path = cache_path.with_suffix(cache_path.suffix + '.tmp')
                torch.save({
                    'chrom': gene.chrom,
                    'gene': gene.name,
                    'tss': gene.tss,
                    'n_snps': n_snps,
                    'positions': torch.as_tensor(selected_positions, dtype=torch.long),
                    'embeddings': selected_embeddings,
                    'variants_by_pos': selected_variants_by_pos,
                }, tmp_path)
                os.replace(tmp_path, cache_path)
                print(f'[{done}/{total}] extracted: {gene.name} {len(selected_positions)} SNPs x {selected_embeddings.shape[1]} dims -> {cache_path}', flush=True)
                del selected_embeddings
        finally:
            del positions
            del embeddings
            load_chrom_embeddings.cache_clear()
            load_chrom_variant_embeddings.cache_clear()
            gc.collect()
    load_gene_embedding_window.cache_clear()
    return sorted(GENE_EMBEDDING_CACHE_DIR.glob('*.pt'))

In [8]:
#cached_gene_embedding_paths = prepare_gene_embedding_cache(genes, N_SNPS)
#print(f'Prepared {len(cached_gene_embedding_paths)} gene embedding windows under {GENE_EMBEDDING_CACHE_DIR}')

In [9]:
@lru_cache(maxsize=1)
def load_major_reference(path: Path) -> dict[tuple[str, int], list[dict[str, str]]]:
    """Map (chrom, pos) to reference major-allele records with REF/ALT identity."""
    df = pd.read_csv(path, sep='\t')
    ref = {}
    for row in df.itertuples(index=False):
        chrom = normalize_chrom(getattr(row, 'CHROM'))
        pos = int(getattr(row, 'POS'))
        ref.setdefault((chrom, pos), []).append({
            'ref': str(getattr(row, 'REF')).upper(),
            'alt': str(getattr(row, 'ALT')).upper(),
            'major_type': str(getattr(row, 'MAJOR_TYPE')).upper(),
        })
    return ref

def open_bigwig_for_sample(sample: str):
    path = bigwig_path_for_sample(sample)
    if not path.exists():
        raise FileNotFoundError(f'Expected normalized bigWig for {sample}: {path}')
    return pyBigWig.open(str(path))

def bw_mean(bw, chrom: str, start: int, end: int) -> float:
    values = np.array(bw.values(chrom, max(0, start), max(0, end), numpy=True), dtype=np.float32)
    if values.size == 0:
        return float('nan')
    values = np.nan_to_num(values, nan=0.0)
    return float(values.mean())

def bw_mean_regions(bw, regions: Iterable[tuple[str, int, int]]) -> float:
    weighted_sum = 0.0
    total_bp = 0
    for chrom, start, end in regions:
        start_i = max(0, int(start))
        end_i = max(start_i, int(end))
        if end_i <= start_i:
            continue
        values = np.array(bw.values(chrom, start_i, end_i, numpy=True), dtype=np.float32)
        if values.size == 0:
            continue
        values = np.nan_to_num(values, nan=0.0)
        weighted_sum += float(values.sum())
        total_bp += int(values.size)
    if total_bp == 0:
        return float('nan')
    return weighted_sum / total_bp

def is_snp_record(record) -> bool:
    if record is None or not record.alts:
        return False
    return len(record.ref) == 1 and all(len(alt) == 1 for alt in record.alts)

def alt_dosage(record, sample_name: str) -> float:
    if not is_snp_record(record):
        return 0.0
    if sample_name not in record.samples:
        raise KeyError(f'{sample_name} not found in VCF record samples.')
    gt = record.samples[sample_name].get('GT')
    if gt is None or any(a is None for a in gt):
        return 0.0
    return float(sum(1 for a in gt if a and a > 0))

def genotype_embedding(record, sample_name: str, variants_at_pos: dict[str, torch.Tensor], embedding_dim: int) -> torch.Tensor:
    vec = torch.zeros(embedding_dim, dtype=torch.float32)
    if not is_snp_record(record) or sample_name not in record.samples:
        return vec
    gt = record.samples[sample_name].get('GT')
    if gt is None:
        return vec
    for allele_index in gt:
        if allele_index is None or allele_index == 0:
            continue
        alt_index = int(allele_index) - 1
        if alt_index < 0 or alt_index >= len(record.alts):
            continue
        key = variant_key(record.ref, record.alts[alt_index])
        emb = variants_at_pos.get(key)
        if emb is not None:
            vec += torch.as_tensor(emb, dtype=torch.float32)
    return vec

def reference_embedding(reference_records: list[dict[str, str]] | None, variants_at_pos: dict[str, torch.Tensor], embedding_dim: int) -> torch.Tensor:
    vec = torch.zeros(embedding_dim, dtype=torch.float32)
    if not reference_records:
        return vec
    for record in reference_records:
        if record.get('major_type') != 'ALT':
            continue
        emb = variants_at_pos.get(variant_key(record['ref'], record['alt']))
        if emb is not None:
            vec += 2.0 * torch.as_tensor(emb, dtype=torch.float32)
    return vec

def sample_name_candidates(sample: str) -> set[str]:
    return {sample, f'CIMA-{sample}', f'CIMA-{sample}_CIMA-{sample}'}

def resolve_vcf_sample_names(vcf: pysam.VariantFile, samples: Iterable[str]) -> dict[str, str]:
    available = set(vcf.header.samples)
    mapping = {}
    for sample in samples:
        matches = sorted(sample_name_candidates(sample) & available)
        if not matches:
            token_matches = [x for x in available if re.search(rf'(^|[-_]){re.escape(sample)}($|[-_])', x)]
            matches = sorted(token_matches)
        if len(matches) != 1:
            raise KeyError(f'Could not uniquely resolve {sample} in VCF samples; matches={matches[:10]}')
        mapping[sample] = matches[0]
    return mapping

def vcf_chrom_name(vcf: pysam.VariantFile, chrom: str) -> str:
    chrom = normalize_chrom(chrom)
    candidates = [chrom, chrom.replace('chr', '', 1)]
    for candidate in candidates:
        if candidate in vcf.header.contigs:
            return candidate
    raise KeyError(f'No contig matching {chrom}; first contigs={list(vcf.header.contigs)[:10]}')

In [10]:
class SNPEmbeddingExpressionDataset(Dataset):
    def __init__(self, samples: list[str], genes: list[GeneRecord], combined_snp_vcf: Path | None, vcf_dir: Path, reference_major_tsv: Path, reference_average_bw: Path):
        self.samples = samples
        self.genes = genes
        self.items = [(sample, gene) for sample in samples for gene in genes]
        self.combined_snp_vcf = Path(combined_snp_vcf) if combined_snp_vcf else None
        self.vcf_dir = Path(vcf_dir)
        self.reference_alt_dosage = load_major_reference(reference_major_tsv)
        self.reference_bw_path = Path(reference_average_bw)
        self._combined_vcf = None
        self._vcfs = {}
        self._sample_maps = {}
        self._reference_bw = None
        self._sample_bw = {}

    def __len__(self):
        return len(self.items)

    @property
    def combined_vcf(self):
        if self.combined_snp_vcf is None:
            return None
        if self._combined_vcf is None:
            self._combined_vcf = pysam.VariantFile(str(self.combined_snp_vcf))
            self._sample_maps.update(resolve_vcf_sample_names(self._combined_vcf, self.samples))
        return self._combined_vcf

    def sample_vcf(self, sample: str):
        if self.combined_snp_vcf is not None:
            return self.combined_vcf
        if sample not in self._vcfs:
            vcf = pysam.VariantFile(str(find_sample_vcf(sample, self.vcf_dir)))
            self._vcfs[sample] = vcf
            self._sample_maps[sample] = resolve_vcf_sample_names(vcf, [sample])[sample]
        return self._vcfs[sample]

    def vcf_sample_name(self, sample: str) -> str:
        if sample not in self._sample_maps:
            self.sample_vcf(sample)
        return self._sample_maps[sample]

    @property
    def reference_bw(self):
        if self._reference_bw is None:
            self._reference_bw = pyBigWig.open(str(self.reference_bw_path))
        return self._reference_bw

    def sample_bw(self, sample: str):
        if sample not in self._sample_bw:
            self._sample_bw[sample] = open_bigwig_for_sample(sample)
        return self._sample_bw[sample]

    def _feature(self, sample: str, gene: GeneRecord) -> torch.Tensor:
        selected_positions, selected_embeddings, variants_by_pos = load_gene_embedding_window(gene, N_SNPS)

        region_start = int(selected_positions.min()) - 1
        region_end = int(selected_positions.max())
        vcf = self.sample_vcf(sample)
        fetch_chrom = vcf_chrom_name(vcf, gene.chrom)
        vcf_sample = self.vcf_sample_name(sample)
        records = {rec.pos: rec for rec in vcf.fetch(fetch_chrom, max(0, region_start), region_end) if is_snp_record(rec)}

        embedding_dim = int(selected_embeddings.shape[1])
        feature = torch.zeros((len(selected_positions), embedding_dim), dtype=torch.float32)
        for i, pos in enumerate(selected_positions):
            pos_i = int(pos)
            rec = records.get(pos_i)
            variants_at_pos = variants_by_pos.get(pos_i, {})
            if variants_at_pos:
                sample_vec = genotype_embedding(rec, vcf_sample, variants_at_pos, embedding_dim)
                ref_vec = reference_embedding(self.reference_alt_dosage.get((gene.chrom, pos_i)), variants_at_pos, embedding_dim)
                feature[i] = sample_vec - ref_vec
            else:
                sample_alt = alt_dosage(rec, vcf_sample) if rec is not None else 0.0
                ref_records = self.reference_alt_dosage.get((gene.chrom, pos_i))
                ref_alt = 2.0 if ref_records and any(r.get('major_type') == 'ALT' for r in ref_records) else 0.0
                feature[i] = selected_embeddings[i] * (sample_alt - ref_alt)

        # Individual-specific feature: exact sample genotype embedding minus train-reference major-allele embedding.
        return feature.reshape(-1)

    def _target(self, sample: str, gene: GeneRecord) -> torch.Tensor:
        sample_value = bw_mean_regions(self.sample_bw(sample), gene.expression_regions)
        reference_value = bw_mean_regions(self.reference_bw, gene.expression_regions)
        return torch.tensor([sample_value - reference_value], dtype=torch.float32)

    def __getitem__(self, index: int):
        sample, gene = self.items[index]
        return {
            'x': self._feature(sample, gene),
            'y': self._target(sample, gene),
            'sample': sample,
            'gene': gene.name,
        }

In [11]:
class ThreeLayerLinearRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden1: int = 2048, hidden2: int = 512, hidden3: int = 128, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.Linear(hidden3, 1),
        )

    def forward(self, x):
        return self.net(x)

def infer_input_dim(first_gene: GeneRecord) -> int:
    _, embeddings, _ = load_gene_embedding_window(first_gene, N_SNPS)
    embedding_dim = int(embeddings.shape[1])
    if EMBEDDING_DIM is not None and int(EMBEDDING_DIM) != embedding_dim:
        warnings.warn(
            f'Configured EMBEDDING_DIM={EMBEDDING_DIM} but actual embedding width is {embedding_dim}; using actual width.'
        )
    return N_SNPS * embedding_dim

In [12]:
train_dataset = SNPEmbeddingExpressionDataset(expression_train_individuals, genes, COMBINED_SNP_VCF, VCF_DIR, REFERENCE_MAJOR_TSV, REFERENCE_AVERAGE_BW)
val_dataset = SNPEmbeddingExpressionDataset(VAL_INDIVIDUALS, genes, COMBINED_SNP_VCF, VCF_DIR, REFERENCE_MAJOR_TSV, REFERENCE_AVERAGE_BW)
test_dataset = SNPEmbeddingExpressionDataset(TEST_INDIVIDUALS, genes, COMBINED_SNP_VCF, VCF_DIR, REFERENCE_MAJOR_TSV, REFERENCE_AVERAGE_BW)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

input_dim = infer_input_dim(genes[0])
inferred_embedding_dim = input_dim // N_SNPS
example_x = train_dataset._feature(expression_train_individuals[0], genes[0])
if int(example_x.numel()) != input_dim:
    raise ValueError(f'Feature dimension mismatch: dataset produced {example_x.numel()} values, model expects {input_dim}.')
model = ThreeLayerLinearRegressor(input_dim=input_dim).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'input_dim={input_dim:,} ({N_SNPS} SNPs x {inferred_embedding_dim} embedding dims)')
print(f'total_params={total_params:,}')
print(f'trainable_params={trainable_params:,}')

swanlab_run = None
if USE_SWANLAB and SWANLAB_MODE != 'disabled':
    if swanlab is None:
        warnings.warn('swanlab is not installed; continuing without SwanLab logging.')
    else:
        try:
            if SWANLAB_API_KEY:
                try:
                    swanlab.login(api_key=SWANLAB_API_KEY)
                except Exception as login_exc:
                    warnings.warn(f'SwanLab login was skipped or failed; continuing to init run: {login_exc}')
            swanlab_run = swanlab.init(
                project=SWANLAB_PROJECT,
                experiment_name=SWANLAB_EXPERIMENT_NAME,
                mode=SWANLAB_MODE,
                config={
                    'n_snps': N_SNPS,
                    'embedding_dim': inferred_embedding_dim,
                    'configured_embedding_dim': EMBEDDING_DIM,
                    'input_dim': input_dim,
                    'target_mode': TARGET_MODE,
                    'target_annotation': str(GENCODE_GTF),
                    'target_region': 'most_3prime_exon',
                    'batch_size': BATCH_SIZE,
                    'epochs': EPOCHS,
                    'lr': LR,
                    'weight_decay': WEIGHT_DECAY,
                    'device': DEVICE,
                    'train_samples': len(expression_train_individuals),
                    'val_samples': len(VAL_INDIVIDUALS),
                    'test_samples': len(TEST_INDIVIDUALS),
                    'n_genes': len(genes),
                    'total_params': total_params,
                    'trainable_params': trainable_params,
                },
            )
        except Exception as exc:
            warnings.warn(f'Failed to initialize SwanLab; continuing without SwanLab logging: {exc}')
            swanlab_run = None

ThreeLayerLinearRegressor(
  (net): Sequential(
    (0): Linear(in_features=524288, out_features=2048, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=2048, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=512, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=1, bias=True)
  )
)
input_dim=524,288 (1024 SNPs x 512 embedding dims)
total_params=1,074,858,753
trainable_params=1,074,858,753


2026-07-10 16:21:15.574 | ERROR    | swanlab.sdk.internal.pkg.safe:block:101 - Error executing 'on_run_initialized'
in callback 'SwanLabLocalCallbacker': dispatcher:65 -> on_run_initialized:346 -> ImportError: Please install 
swanboard to use 'local' mode: pip install 'swanlab[dashboard]'

swanlab: Tracking run with swanlab version 0.8.3

swanlab: 💾 Run data saved at 
/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/swanlog/run-20260710_162114-2s4qrj8g

swanlab: 🌟 Run `swanlab watch /mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/swanlog` to view SwanLab 
Experiment Dashboard

In [ ]:
def move_batch(batch, device: str):
    return batch['x'].to(device), batch['y'].to(device)

def safe_pearson(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return float('nan')
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    if ss_tot == 0:
        return float('nan')
    return float(1 - ss_res / ss_tot)

def summarize_group_metrics(predictions: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for group_value, df in predictions.groupby(group_col, sort=True):
        y_true = df['target_diff'].to_numpy(dtype=float)
        y_pred = df['prediction_diff'].to_numpy(dtype=float)
        rows.append({
            group_col: group_value,
            'n': int(len(df)),
            'pearson': safe_pearson(y_true, y_pred),
            'r2': safe_r2(y_true, y_pred),
            'rmse': float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
            'mae': float(np.mean(np.abs(y_pred - y_true))),
        })
    return pd.DataFrame(rows)

def metric_mean(values) -> float:
    values = pd.Series(values, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(values.mean()) if len(values) else float('nan')

def metric_median(values) -> float:
    values = pd.Series(values, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(values.median()) if len(values) else float('nan')

def swanlab_safe_log(data: dict[str, float], step: int):
    if swanlab_run is None or swanlab is None:
        return
    clean = {key: value for key, value in data.items() if value is not None and np.isfinite(float(value))}
    if not clean:
        return
    try:
        swanlab.log(clean, step=step)
    except Exception as exc:
        warnings.warn(f'Failed to log metrics to SwanLab at step {step}: {exc}')

def swanlab_metric_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))

def run_epoch(model, loader, optimizer=None, return_predictions: bool = False):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    n = 0
    preds = []
    targets = []
    rows = []
    with torch.set_grad_enabled(train_mode):
        for batch in loader:
            x, y = move_batch(batch, DEVICE)
            pred = model(x)
            loss = criterion(pred, y)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            batch_size = x.shape[0]
            total_loss += float(loss.detach().cpu()) * batch_size
            n += batch_size
            preds.append(pred.detach().cpu())
            targets.append(y.detach().cpu())
            if return_predictions:
                batch_preds = pred.detach().cpu().numpy().reshape(-1)
                batch_targets = y.detach().cpu().numpy().reshape(-1)
                for sample, gene, p, t in zip(batch['sample'], batch['gene'], batch_preds, batch_targets):
                    rows.append({'sample': sample, 'gene': gene, 'prediction_diff': float(p), 'target_diff': float(t)})
    preds = torch.cat(preds).numpy().reshape(-1)
    targets = torch.cat(targets).numpy().reshape(-1)
    rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
    mae = float(np.mean(np.abs(preds - targets)))
    metrics = {
        'loss': total_loss / max(n, 1),
        'rmse': rmse,
        'mae': mae,
        'pearson': safe_pearson(targets, preds),
        'r2': safe_r2(targets, preds),
    }
    if return_predictions:
        return metrics, pd.DataFrame(rows)
    return metrics

best_val = math.inf
best_path = PROJECT_DIR / 'best_snp_embedding_expression_model.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer)
    val_metrics, val_predictions = run_epoch(model, val_loader, return_predictions=True)
    val_metrics_by_gene = summarize_group_metrics(val_predictions, 'gene')
    row = {
        'epoch': epoch,
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'val_gene_pearson_mean': metric_mean(val_metrics_by_gene['pearson']),
        'val_gene_pearson_median': metric_median(val_metrics_by_gene['pearson']),
        'val_gene_r2_mean': metric_mean(val_metrics_by_gene['r2']),
        'val_gene_r2_median': metric_median(val_metrics_by_gene['r2']),
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(row)
    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        torch.save({'model_state_dict': model.state_dict(), 'input_dim': input_dim, 'config': row}, best_path)
    log_row = {
        'train/loss': train_metrics['loss'],
        'train/rmse': train_metrics['rmse'],
        'train/mae': train_metrics['mae'],
        'train/pearson': train_metrics['pearson'],
        'train/r2': train_metrics['r2'],
        'val/loss': val_metrics['loss'],
        'val/rmse': val_metrics['rmse'],
        'val/mae': val_metrics['mae'],
        'val/pearson': val_metrics['pearson'],
        'val/r2': val_metrics['r2'],
        'val_by_gene/pearson_mean': row['val_gene_pearson_mean'],
        'val_by_gene/pearson_median': row['val_gene_pearson_median'],
        'val_by_gene/r2_mean': row['val_gene_r2_mean'],
        'val_by_gene/r2_median': row['val_gene_r2_median'],
        'lr': row['lr'],
        'best_val_loss': best_val,
    }
    if SWANLAB_LOG_PER_GENE_VAL:
        for metric_row in val_metrics_by_gene.itertuples(index=False):
            gene = swanlab_metric_name(getattr(metric_row, 'gene'))
            log_row[f'val_by_gene/{gene}/pearson'] = getattr(metric_row, 'pearson')
            log_row[f'val_by_gene/{gene}/r2'] = getattr(metric_row, 'r2')
    swanlab_safe_log(log_row, step=epoch)

pd.DataFrame(history).to_csv(PROJECT_DIR / 'training_history.csv', index=False)

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
test_metrics = run_epoch(model, test_loader)
test_metrics

In [ ]:
def collect_predictions(model, loader) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            x, y = move_batch(batch, DEVICE)
            pred = model(x).detach().cpu().numpy().reshape(-1)
            y = y.detach().cpu().numpy().reshape(-1)
            for sample, gene, p, t in zip(batch['sample'], batch['gene'], pred, y):
                rows.append({'sample': sample, 'gene': gene, 'prediction_diff': float(p), 'target_diff': float(t)})
    return pd.DataFrame(rows)

predictions = collect_predictions(model, test_loader)
predictions.to_csv(PROJECT_DIR / 'test_predictions.csv', index=False)
predictions

In [ ]:
def safe_pearson(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0:
        return float('nan')
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    if ss_tot == 0:
        return float('nan')
    return float(1 - ss_res / ss_tot)

def summarize_group_metrics(predictions: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for group_value, df in predictions.groupby(group_col, sort=True):
        y_true = df['target_diff'].to_numpy(dtype=float)
        y_pred = df['prediction_diff'].to_numpy(dtype=float)
        rows.append({
            group_col: group_value,
            'n': int(len(df)),
            'pearson': safe_pearson(y_true, y_pred),
            'r2': safe_r2(y_true, y_pred),
            'rmse': float(np.sqrt(np.mean((y_pred - y_true) ** 2))),
            'mae': float(np.mean(np.abs(y_pred - y_true))),
        })
    return pd.DataFrame(rows)

if 'predictions' not in globals():
    predictions = pd.read_csv(PROJECT_DIR / 'test_predictions.csv')

metrics_by_individual = summarize_group_metrics(predictions, 'sample')
metrics_by_gene = summarize_group_metrics(predictions, 'gene')

metrics_by_individual.to_csv(PROJECT_DIR / 'test_metrics_by_individual.csv', index=False)
metrics_by_gene.to_csv(PROJECT_DIR / 'test_metrics_by_gene.csv', index=False)

display(metrics_by_individual)
display(metrics_by_gene)
